In [82]:
import pandas as pd
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder

In [74]:
train = pd.concat([
    pd.read_excel("训练集6.5.xlsx", sheet_name="1").T,
    pd.read_excel("训练集6.5.xlsx", sheet_name="2").T,
    pd.read_excel("训练集6.5.xlsx", sheet_name="3").T,
    pd.read_excel("训练集6.5.xlsx", sheet_name="4").T,
])
train = train.reset_index(drop=True)
train.columns = train.iloc[0]
train = train[train["水/硅源摩尔比"] != "水/硅源摩尔比"]

In [75]:
test = pd.read_excel("测试集6.5.xlsx").T
test = test.reset_index(drop=True)
test.columns = test.iloc[0]
test = test.iloc[1:]

In [76]:
train["水/硅源摩尔比"] = train["水/硅源摩尔比"].astype(float)
test["水/硅源摩尔比"] = test["水/硅源摩尔比"].astype(float)

train["pHa"] = train["pHa"].astype(float)
test["pHa"] = test["pHa"].astype(float)

train["pHb"] = train["pHb"].astype(float)
test["pHb"] = test["pHb"].astype(float)

train["老化时间"] = train["老化时间"].astype(float)
test["老化时间"] = test["老化时间"].astype(float)

for col in ['密度（kg/m³）', '比表面积（m²/g）', '平均孔径（nm）', '热导率（W/m·K）', '抗压强度（MPa）']:
    train[col] = train[col].astype(float)

In [77]:
train.dtypes

0
硅源类型（TEOS / TMOS / MTMS等）     object
溶剂类型（EtOH / MeOH等）            object
水/硅源摩尔比                      float64
催化剂类型（酸/碱）                    object
pHa                          float64
pHb                          float64
老化时间                         float64
干燥方式（常压/超临界/冷冻）               object
密度（kg/m³）                    float64
比表面积（m²/g）                   float64
平均孔径（nm）                     float64
热导率（W/m·K）                   float64
抗压强度（MPa）                    float64
dtype: object

In [78]:
test.dtypes

0
硅源类型（TEOS / TMOS / MTMS等）     object
溶剂类型（EtOH / MeOH等）            object
水/硅源摩尔比                      float64
催化剂类型（酸/碱）                    object
pHa                          float64
pHb                          float64
老化时间                         float64
干燥方式（常压/超临界/冷冻）               object
dtype: object

In [79]:
train.columns

Index(['硅源类型（TEOS / TMOS / MTMS等）', '溶剂类型（EtOH / MeOH等）', '水/硅源摩尔比',
       '催化剂类型（酸/碱）', 'pHa', 'pHb', '老化时间', '干燥方式（常压/超临界/冷冻）', '密度（kg/m³）',
       '比表面积（m²/g）', '平均孔径（nm）', '热导率（W/m·K）', '抗压强度（MPa）'],
      dtype='object', name=0)

In [80]:
test.columns

Index(['硅源类型（TEOS / TMOS / MTMS等）', '溶剂类型（EtOH / MeOH等）', '水/硅源摩尔比',
       '催化剂类型（酸/碱）', 'pHa', 'pHb', '老化时间', '干燥方式（常压/超临界/冷冻）'],
      dtype='object', name=0)

In [81]:
for col in train.select_dtypes("object").columns:
    lbl = LabelEncoder()
    lbl.fit(list(train[col]) + list(test[col]))
    train[col] = lbl.transform(train[[col]])
    test[col] = lbl.transform(test[[col]])

/home/lyz/anaconda3/envs/py311/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/lyz/anaconda3/envs/py311/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/lyz/anaconda3/envs/py311/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/lyz/anaconda3/envs/py311/lib/python3.11/site-packages/s

In [84]:
model = XGBRegressor()
model.fit(
    train.drop(['密度（kg/m³）', '比表面积（m²/g）', '平均孔径（nm）', '热导率（W/m·K）', '抗压强度（MPa）'], axis=1),
    train[['密度（kg/m³）', '比表面积（m²/g）', '平均孔径（nm）', '热导率（W/m·K）', '抗压强度（MPa）']]
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [96]:
submit = model.predict(test)
submit = pd.DataFrame(submit)
submit.columns = ["density", "surface_area", "pore_diameter", "thermal_conductivity", "compressive_strength"]
submit["id"] = range(1, 101)
submit = submit[["id", "density", "surface_area", "pore_diameter", "thermal_conductivity", "compressive_strength"]]

In [95]:
!mkdir submit

In [ ]:
submit.to_csv("submit/")